# 1: Downloading the ASL Alphabet Dataset from Kaggle

In [ ]:
import kagglehub
dataset_path = kagglehub.dataset_download('grassknoted/asl-alphabet')

Using Colab cache for faster access to the 'asl-alphabet' dataset.


# 2: Installing the split-folders Library

In [ ]:
pip install split-folders

In [ ]:
pip install mediapipe

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 10.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: opencv-contrib-python
    Found existing installation: opencv-contrib-python 4.12.0.88
    Uninstalling openc

# 3: Importing Libraries and Dependencies

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random

import cv2
import mediapipe as mp
from tqdm import tqdm

import splitfolders

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from keras.models import Sequential
from keras.layers import Dense,Conv2D,Dropout,Flatten,MaxPooling2D, BatchNormalization,Input,concatenate
from keras.callbacks import EarlyStopping,ReduceLROnPlateau, ModelCheckpoint
from keras.utils import plot_model

from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

from google.colab import files
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
from tensorflow.keras import layers
from tensorflow.keras.models import Model


# 4: Exploring the Dataset Directory

In [ ]:
print(f"\nContents of {dataset_path}:")
print(os.listdir(dataset_path))

# Check if there are subdirectories
for item in os.listdir(dataset_path):
    item_path = os.path.join(dataset_path, item)
    if os.path.isdir(item_path):
        print(f"\nContents of folder '{item}':")
        print(os.listdir(item_path)[:5])

# 5: Crop all images in the dataset to focus on hand using mediapipe

In [ ]:
# Paths
DATASET_DIR = "/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train"
OUTPUT_DIR = "/kaggle/working/asl_alphabet_cropped"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mediapipe hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1)
mp_draw = mp.solutions.drawing_utils

IMG_SIZE = 224  # crop target size

def crop_hand(img):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if not results.multi_hand_landmarks:
        return None  # no hand detected

    h, w, _ = img.shape
    lm = results.multi_hand_landmarks[0]

    xs = [p.x for p in lm.landmark]
    ys = [p.y for p in lm.landmark]
    min_x, max_x = int(min(xs)*w), int(max(xs)*w)
    min_y, max_y = int(min(ys)*h), int(max(ys)*h)

    # expand box a little
    margin = 20
    min_x, min_y = max(min_x - margin, 0), max(min_y - margin, 0)
    max_x, max_y = min(max_x + margin, w), min(max_y + margin, h)

    cropped = img[min_y:max_y, min_x:max_x]
    if cropped.size == 0:
        return None

    cropped = cv2.resize(cropped, (IMG_SIZE, IMG_SIZE))
    return cropped

# Process dataset
for class_name in tqdm(os.listdir(DATASET_DIR)):
    class_path = os.path.join(DATASET_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    save_class_path = os.path.join(OUTPUT_DIR, class_name)
    os.makedirs(save_class_path, exist_ok=True)

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue

        cropped = crop_hand(img)
        if cropped is not None:
            cv2.imwrite(os.path.join(save_class_path, img_name), cropped)


  3%|▎         | 1/29 [02:21<1:06:15, 142.00s/it]


KeyboardInterrupt: 